In [12]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from IPython.display import display, Math


In [8]:
rho, u, E ,gamma,p= sp.symbols('rho u E gamma p')

In [ ]:
N=40
XMIN = -10
XMAX = 10
X = np.linspace(XMIN, XMAX, N)
DX= (XMAX - XMIN) / (N-1)

GAMMA = 1.4

}

In [ ]:
class Derive(rho, u, E, gamma):
    def __init__(self, rho, u, p, gamma):
        self.rho = rho
        self.u = u
        self.gamma = gamma
        self.p = p
        self.E =  p/((gamma-1)*rho)  +0.5*u**2
        
        
    def U_to_primative(self, U):
        rho = U[0]
        u = U[1]/U[0]
        E = U[2]
        p = (self.gamma - 1) * (E - rho * u**2 / 2)
        return sp.Matrix([rho, u, p])
    

In [ ]:
CFL = 0.4
T_START = 0.0
T_END = 0.001

N=40
XMIN = -10
XMAX = 10

WL = {
    'rho': 1.0,
    'u': 0.0,
    'p': 100000.0
}

WR = {
    'rho': 0.125,
    'u': 0.0,
    'p': 10000.0
}


In [ ]:

class Solver:
    def __init__(self, scheme, xmin, xmax, N,wl, wr, gamma=1.4, CFL=0.4):
        
        self.flux = scheme
        self.gamma = gamma
        self.CFL = CFL

        self._build_grid(xmin, xmax, N)
        self._build_IC(wl, wr)
        
        self.U_new = np.zeros_like(self.U)
        self.F = np.zeros((N-1,3))
    def _build_grid(self, xmin, xmax, N):
        self.N = N
        self.x = np.linspace(xmin, xmax, N)
        self.dx = (xmax - xmin) / (N-1)
        return 
    def _build_IC(self, wl, wr):
        self.U = np.zeros((self.N,3))
        for i in range(self.N):
            w = wl if self.x[i] < 0 else wr 
            rho, u, p = (w[var] for var in ('rho', 'u', 'p'))
            E = p/((self.gamma-1)*rho) + 0.5*u**2
            self.U[i] = [rho, rho*u, rho*E]
        return
    def run(self, t_start, t_end):
        t = t_start
        while t < t_end:
            dt = self.calc_dt()
            self.F =self.calc_flux()
            self.U = self.calc_u()
            t += dt
    def calc_flux(self):
        F=np.zeros((N-1, 3))
        for i in range(self.N-1):
            UL = self.U[i]
            UR = self.U[i+1] 
            rho, u, E, p, a = self.U_to_primative(self.U[i])
            Fp = self.build_F(rho, u, a, self.fp(u,a))
            Fm = self.build_F(rho, u, a, self.fm(u,a))
            if self.flux == "sw":
                F[i]= Fp + Fm
            else:
                print("Unknown flux scheme")
                break
        return F
    def calc_u(self):
        U_new = self.U.copy()
        for i in range(1, self.N-1):
            self.U_new[i] = self.U[i] - (self.F[i] - self.F[i-1]) * self.dx
        return U_new
    def calc_dt(self):
        max_speed = 0.0
        for i in range(self.N):
            _, _, u, _, a = self.U_to_primative(self.U[i])
            max_speed = max(max_speed, abs(u) + a)
            dt = self.CFL * self.dx / max_speed
        return dt
    def lp(self, u, a):
        return [max(0, u-a), max(0, u), max(0, u+a)]
    def lm(self, u, a):
        return [min(0, u-a), min(0, u), min(0, u+a)]
    def build_F(self, rho, u,a, L):
        gamma = self.gamma
        l1, l2, l3 = L
        F1 = 2*(gamma-1)*l1 + l2 + l3
        F2 = 2*(gamma-1)*l1*u + l2*(u+a) + l3*(u-a)
        F3 = ((gamma-1)*l1*u**2
            + 0.5*l2*(u+a)**2
            + 0.5*l3*(u-a)**2
            + (3-gamma)/(2*(gamma-1))*(l2+l3)*a**2)
        return (rho/(2*gamma)) * np.array([F1, F2, F3])
    def U_to_primative(self, U):
        rho = U[0]
        u   = U[1]/rho
        E   = U[2]/rho
        p   = (self.gamma-1)*(rho*E - 0.5*rho*u**2)
        a   = np.sqrt(self.gamma*p/rho)
        return rho, u, E, p, a


Solver("sw", XMIN, XMAX, N, WL, WR, CFL=CFL)


In [ ]:
class StegerWarming:
    def __init__(self, lambda_to_flux):
        self.lambda_to_flux = lambda_to_flux

    def build(self, primL, primR):
        rhoL, uL, _, aL = primL
        rhoR, uR, _, aR = primR

        Lp = [max(uL,0), max(uL+aL,0), max(uL-aL,0)]
        Lm = [min(uR,0), min(uR+aR,0), min(uR-aR,0)]

        Fp = self.lambda_to_flux(rhoL, uL, aL, *Lp)
        Fm = self.lambda_to_flux(rhoR, uR, aR, *Lm)

        return Fp + Fm


In [ ]:
class UDS:

    def build(self, phi, flux_cv, i, j):

        phiP = phi[i,j]
        phiE = phi[i+1,j]
        phiW = phi[i-1,j]
        phiN = phi[i,j+1]
        phiS = phi[i,j-1]

        Fe = flux_cv.e[0]
        Fw = flux_cv.w[0]
        Fn = flux_cv.n[0]
        Fs = flux_cv.s[0]

        return (
            phiP*sp.Max(Fe,0.0) - phiE*sp.Max(-Fe,0.0)
            - phiW*sp.Max(Fw,0.0) + phiP*sp.Max(-Fw,0.0)
            + phiP*sp.Max(Fn,0.0) - phiN*sp.Max(-Fn,0.0)
            - phiS*sp.Max(Fs,0.0) + phiP*sp.Max(-Fs,0.0)
        )
